
## Real-Time Streaming Feature Engineering for Credit Card Fraud Detection


Credit card fraud costs the global economy over [**$30 billion annually**](https://www.globenewswire.com/news-release/2026/01/07/3214821/0/en/Global-Card-Fraud-Losses-at-33-Billion.html), and the window to detect a fraudulent transaction is measured in milliseconds — not minutes or hours. Traditional batch-based feature pipelines introduce latency that renders fraud signals stale before they ever reach a model. By the time a batch pipeline computes that a user has made 10 transactions in 5 minutes across 3 countries, the damage is already done.

**This solution demonstrates how to close that gap** using new capabilities on Databricks:

- **[Real-Time Mode (RTM)](https://www.databricks.com/blog/introducing-real-time-mode-apache-sparktm-structured-streaming)** — A new execution mode for Apache Spark Structured Streaming that eliminates micro-batch scheduling overhead, enabling **continuous, sub-second processing** of streaming events. Unlike traditional micro-batch triggers, RTM processes each record as soon as it arrives, dramatically reducing end-to-end latency.

- **[`transformWithState`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)** — A new stateful processing API introduced in Spark 4.0 that provides fine-grained control over per-key state management. It replaces the legacy `mapGroupsWithState`/`flatMapGroupsWithState` APIs with a cleaner, more powerful interface supporting typed state variables, TTL-based expiration, and RocksDB-backed persistence with changelog checkpointing.



This notebook handles the initial setup and configuration for the **Real-Time streaming feature engineering pipeline** that computes fraud detection features from credit card transactions and publishes them to [Databricks Lakebase](https://docs.databricks.com/en/lakebase/index.html) PostgreSQL with sub-second latency.

The pipeline consists of three notebooks that should be run in order:

- **00_setup** (this notebook) — Validate prerequisites, connect to Lakebase, and create the feature table
- **01_generate_streaming_data** — Generate synthetic credit card transactions and publish to Kafka
- **02_streaming_fraud_detection_pipeline** — Read from Kafka, compute stateless and stateful features, write to Lakebase

> Run this notebook **once** before starting the streaming pipeline. It is safe to re-run — the table creation is idempotent.


<!-- <img src="./images/streaming_fraud_setup_overview.png" alt="Setup Overview" width="800px" style="display: block; margin: 0 auto 20px auto"/> -->


<div style="font-family:'Segoe UI',system-ui,-apple-system,sans-serif;background:linear-gradient(135deg,#0a1628 0%,#1a2744 40%,#0d2137 100%);border-radius:16px;padding:36px 40px 30px;color:#fff;position:relative;overflow:hidden;">
  <div style="position:absolute;top:-50%;right:-20%;width:500px;height:500px;background:radial-gradient(circle,rgba(56,189,248,0.08) 0%,transparent 70%);pointer-events:none;"></div>
  <div style="text-align:center;margin-bottom:8px;font-size:24px;font-weight:600;color:#e2e8f0;letter-spacing:-0.5px;">Setup &amp; Configuration</div>
  <div style="display:flex;align-items:stretch;justify-content:center;gap:12px;flex-wrap:wrap;">
    <!-- Phase 1 -->
    <div style="flex:1;min-width:180px;max-width:240px;background:linear-gradient(180deg,rgba(37,99,235,0.15) 0%,rgba(37,99,235,0.04) 100%);border:1px solid rgba(96,165,250,0.2);border-radius:14px;padding:22px 18px;position:relative;">
      <div style="position:absolute;top:10px;right:14px;font-size:10px;font-weight:700;color:rgba(255,255,255,0.25);letter-spacing:1px;">PHASE 1</div>
      <div style="width:48px;height:48px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-size:22px;background:linear-gradient(135deg,#1e3a5f,#2563eb);margin-bottom:12px;">📋</div>
      <div style="font-size:15px;font-weight:700;color:#60a5fa;margin-bottom:8px;">Prerequisites</div>
      <div style="font-size:11px;color:#94a3b8;line-height:1.7;">
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#3b82f6;flex-shrink:0;display:inline-block;"></span> databricks-sdk ≥ 0.65.0</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#3b82f6;flex-shrink:0;display:inline-block;"></span> dbldatagen library</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#3b82f6;flex-shrink:0;display:inline-block;"></span> DBR 17.3+ / Spark 4.0+</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#3b82f6;flex-shrink:0;display:inline-block;"></span> Real-Time cluster config</div>
      </div>
    </div>
    <!-- Arrow 1 -->
    <div style="display:flex;align-items:center;justify-content:center;flex-shrink:0;">
      <svg width="40" height="24" viewBox="0 0 40 24"><line x1="0" y1="12" x2="28" y2="12" stroke="#818cf8" stroke-width="2" stroke-dasharray="6 4" opacity="0.6"/><polygon points="26,6 40,12 26,18" fill="#818cf8" opacity="0.7"/></svg>
    </div>
    <!-- Phase 2 -->
    <div style="flex:1;min-width:180px;max-width:240px;background:linear-gradient(180deg,rgba(13,148,136,0.15) 0%,rgba(13,148,136,0.04) 100%);border:1px solid rgba(45,212,191,0.2);border-radius:14px;padding:22px 18px;position:relative;">
      <div style="position:absolute;top:10px;right:14px;font-size:10px;font-weight:700;color:rgba(255,255,255,0.25);letter-spacing:1px;">PHASE 2</div>
      <div style="width:48px;height:48px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-size:22px;background:linear-gradient(135deg,#134e4a,#0d9488);margin-bottom:12px;">🔌</div>
      <div style="font-size:15px;font-weight:700;color:#2dd4bf;margin-bottom:8px;">Lakebase Connection</div>
      <div style="font-size:11px;color:#94a3b8;line-height:1.7;">
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#14b8a6;flex-shrink:0;display:inline-block;"></span> LakebaseClient init</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#14b8a6;flex-shrink:0;display:inline-block;"></span> OAuth credentials</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#14b8a6;flex-shrink:0;display:inline-block;"></span> psycopg2 SSL connect</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#14b8a6;flex-shrink:0;display:inline-block;"></span> Connection test &amp; verify</div>
      </div>
    </div>
    <!-- Arrow 2 -->
    <div style="display:flex;align-items:center;justify-content:center;flex-shrink:0;">
      <svg width="40" height="24" viewBox="0 0 40 24"><line x1="0" y1="12" x2="28" y2="12" stroke="#818cf8" stroke-width="2" stroke-dasharray="6 4" opacity="0.6"/><polygon points="26,6 40,12 26,18" fill="#818cf8" opacity="0.7"/></svg>
    </div>
    <!-- Phase 3 -->
    <div style="flex:1;min-width:180px;max-width:240px;background:linear-gradient(180deg,rgba(217,119,6,0.15) 0%,rgba(217,119,6,0.04) 100%);border:1px solid rgba(251,191,36,0.2);border-radius:14px;padding:22px 18px;position:relative;">
      <div style="position:absolute;top:10px;right:14px;font-size:10px;font-weight:700;color:rgba(255,255,255,0.25);letter-spacing:1px;">PHASE 3</div>
      <div style="width:48px;height:48px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-size:22px;background:linear-gradient(135deg,#713f12,#d97706);margin-bottom:12px;">🗄️</div>
      <div style="font-size:15px;font-weight:700;color:#fbbf24;margin-bottom:8px;">Schema Creation</div>
      <div style="font-size:11px;color:#94a3b8;line-height:1.7;">
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#f59e0b;flex-shrink:0;display:inline-block;"></span> CREATE TABLE IF NOT EXISTS</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#f59e0b;flex-shrink:0;display:inline-block;"></span> 48 feature columns</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#f59e0b;flex-shrink:0;display:inline-block;"></span> 7 optimized indexes</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#f59e0b;flex-shrink:0;display:inline-block;"></span> PK: transaction_id</div>
      </div>
    </div>
    <!-- Arrow 3 -->
    <div style="display:flex;align-items:center;justify-content:center;flex-shrink:0;">
      <svg width="40" height="24" viewBox="0 0 40 24"><line x1="0" y1="12" x2="28" y2="12" stroke="#818cf8" stroke-width="2" stroke-dasharray="6 4" opacity="0.6"/><polygon points="26,6 40,12 26,18" fill="#818cf8" opacity="0.7"/></svg>
    </div>
    <!-- Phase 4 -->
    <div style="flex:1;min-width:180px;max-width:240px;background:linear-gradient(180deg,rgba(22,163,74,0.15) 0%,rgba(22,163,74,0.04) 100%);border:1px solid rgba(74,222,128,0.2);border-radius:14px;padding:22px 18px;position:relative;">
      <div style="position:absolute;top:10px;right:14px;font-size:10px;font-weight:700;color:rgba(255,255,255,0.25);letter-spacing:1px;">PHASE 4</div>
      <div style="width:48px;height:48px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-size:22px;background:linear-gradient(135deg,#14532d,#16a34a);margin-bottom:12px;">✅</div>
      <div style="font-size:15px;font-weight:700;color:#4ade80;margin-bottom:8px;">Verification</div>
      <div style="font-size:11px;color:#94a3b8;line-height:1.7;">
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#22c55e;flex-shrink:0;display:inline-block;"></span> Table existence check</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#22c55e;flex-shrink:0;display:inline-block;"></span> Row count validation</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#22c55e;flex-shrink:0;display:inline-block;"></span> Schema confirmation</div>
        <div style="display:flex;align-items:center;gap:6px;"><span style="width:5px;height:5px;border-radius:50%;background:#22c55e;flex-shrink:0;display:inline-block;"></span> Ready for pipeline</div>
      </div>
    </div>
  </div>

</div>


## Prerequisites

Before running this notebook, ensure the following:

- **Databricks Runtime 17.3+** (with Spark 4.0+ for `transformWithState`)
- Cluster configured for [Real-Time Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#cluster-configuration) with sufficient [task slots/cores](https://docs.databricks.com/aws/en/structured-streaming/real-time#cluster-size-requirements)
- **Databricks Python SDK >= 0.65.0** installed on the cluster
- **dbldatagen** library installed on the cluster
- Access to an existing **Lakebase PostgreSQL** instance (configure in `utils/config.py`)

| Output Table | Description |
|---|---|
| `transaction_features` | Stores both stateless and stateful fraud detection features |


## 1: Import Libraries and Configure Logging

Import the core PySpark and Python libraries needed for setup operations.

In [0]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


## 2: Validate Databricks SDK

Verify that `databricks-sdk >= 0.65.0` is installed. This version is required for Lakebase SDK support.

In [0]:
#Validate if databricks-sdk > 0.65.0 is installed to support Lakebase SDK
%pip show databricks-sdk | grep -oP '(?<=Version: )\S+'


## 3: Validate dbldatagen Library

Verify that the `dbldatagen` library is installed. This library generates realistic synthetic credit card transaction data in the data generator notebook.

In [0]:
#Validate if dbldatagen is installed for kafka data generation
import dbldatagen as dg
print("dbldatagen version:", dg.__version__)


## 4: Connect to Lakebase and Create Feature Table


This step performs the following:

1. Initialize the `LakebaseClient` using configuration from `utils/config.py`
2. Test the PostgreSQL connection
3. Create the `transaction_features` table for storing stateless and stateful fraud features
4. Verify the table was created successfully

> 💡 The `LakebaseClient` handles authentication automatically using the Databricks SDK. See `utils/lakebase_client.py` for implementation details.

<img src="./images/streaming_fraud_lakebase_setup.png" alt="Lakebase Setup Flow" style="display: block; margin: 0 auto 20px auto; width: 100%"/>

In [0]:
# Import Lakebase client
from utils.lakebase_client import LakebaseClient
from utils.config import Config

#initialize Config
config = Config()

print("Connecting to Lakebase PostgreSQL...\n")

# Initialize Lakebase client
lakebase = LakebaseClient(**config.lakebase_config)

# Test connection
print("Testing Lakebase connection...")
if lakebase.test_connection():
    print("Successfully connected to Lakebase PostgreSQL!")
else:
    print("Failed to connect to Lakebase")
    print("  Please check:")
    print("  1. Lakebase instance is provisioned")
    print("  2. Instance name is correct")
    print("  3. Database name is correct")
    raise Exception("Lakebase connection failed")

# Create unified feature table
print("\nCreating unified feature table in Lakebase...")
print("  Table: transaction_features")
print("  Includes: stateless + stateful fraud detection features")

lakebase.create_feature_table("transaction_features")

print("Table created successfully!")


## 5: Verification

Confirm that the `transaction_features` table exists in Lakebase and is ready to receive data from the streaming pipeline.

In [0]:
# Verify table exists
print("Verifying table...")
try:
    stats_txn = lakebase.get_table_stats("transaction_features")
    print(f"  transaction_features: {stats_txn['total_rows']:,} rows")
except Exception as e:
    print("  Table exists but is empty (just created)")

print("\n" + "="*60)
print("LAKEBASE POSTGRESQL SETUP COMPLETE")
print("="*60)


## Next Steps

Setup is complete. Proceed with the following notebooks in order:

1. Open the [01_generate_streaming_data notebook]($./01_generate_streaming_data) to generate synthetic credit card transactions and publish them to Kafka
2. Open the [02_streaming_fraud_detection_pipeline notebook]($./02_streaming_fraud_detection_pipeline) to run the real-time feature engineering pipeline